In [3]:
import torch

# Your defined parameters
T, C, N, S = 50, 20, 16, 30

# 1. The Input (Log-Probs)
input = torch.randn(T, N, C).log_softmax(2).detach().requires_grad_()

# 2. The Targets (Random class indices between 1 and C-1)
# Note: Target indices should not include the blank (usually index 0)
target = torch.randint(low=1, high=C, size=(N, S), dtype=torch.long)

# 3. The Lengths
input_lengths = torch.full(size=(N,), fill_value=T, dtype=torch.long)
target_lengths = torch.randint(low=10, high=S, size=(N,), dtype=torch.long)

# 4. The Loss
ctc_loss = torch.nn.CTCLoss()
loss = ctc_loss(input, target, input_lengths, target_lengths)
# input.shape

In [2]:
# count number of videos in data/How2Sign/sentence_level/val/rgb_front/features/openpose_output/video
from pathlib import Path

video_dir = Path("data/How2Sign/sentence_level/val/rgb_front/features/openpose_output/video")
video_extensions = {".mp4"}
videos = [f for f in video_dir.iterdir() if f.is_file() and f.suffix.lower() in video_extensions]
print(f"Number of videos: {len(videos)}")


Number of videos: 2343


In [4]:
# data/How2Sign/sentence_level/val/text/en/raw_text/re_aligned/how2sign_realigned_val.csv
import csv

# Path to the CSV file
csv_path = Path("data/How2Sign/sentence_level/val/text/en/raw_text/re_aligned/how2sign_realigned_val.csv")

# Get all video base names (without extension) from the folder
video_folder_names = set(f.stem for f in videos)

# Get all VIDEO_NAMEs from the CSV
csv_video_names = set()

with open(csv_path, 'r', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile, delimiter='\t')
    for row in reader:
        # VIDEO_NAME column in the CSV
        csv_video_names.add(row['VIDEO_NAME'])

# Compare sets
print("Number of unique videos in folder:", len(video_folder_names))
print("Number of unique VIDEO_NAMEs in CSV:", len(csv_video_names))

missing_in_folder = csv_video_names - video_folder_names
missing_in_csv = video_folder_names - csv_video_names

if not missing_in_folder and not missing_in_csv:
    print("All video files match the CSV VIDEO_NAMEs.")
else:
    if missing_in_folder:
        print("These VIDEO_NAMEs are in CSV but missing in folder:", missing_in_folder)
    if missing_in_csv:
        print("These files are in folder but not in CSV VIDEO_NAMEs:", missing_in_csv)

Number of unique videos in folder: 2343
Number of unique VIDEO_NAMEs in CSV: 132
These VIDEO_NAMEs are in CSV but missing in folder: {'1aJwX9nRlmk-2-rgb_front', '39FN42e41r0-1-rgb_front', 'CzkLI34HFIg-5-rgb_front', 'dIIMHOX5AD8-8-rgb_front', 'cHFW_U9e4sM-8-rgb_front', 'ETOZLBScxWY-5-rgb_front', '4CSSlWonj3E-2-rgb_front', '5Gw0TpbCMTQ-5-rgb_front', '0zvsqf23tmw-2-rgb_front', '1V-aOg8wCT4-1-rgb_front', 'afDfb8xx09w-3-rgb_front', '0pKzG0RRUz4-1-rgb_front', 'abzRFn8xngA-3-rgb_front', '2yudAtTnZrg-1-rgb_front', 'FMSiqrDXABU-3-rgb_front', '48hXSSOwsNE-2-rgb_front', 'EhYk6e-jtWg-3-rgb_front', '33YRZ5UZTA4-2-rgb_front', '46Cwjrd4ua4-2-rgb_front', 'fE6xxSbjVV8-8-rgb_front', 'cw5evdziBB4-8-rgb_front', 'EQWFrWeRVjQ-5-rgb_front', 'BL4ZqiUZO1U-5-rgb_front', '1G8LIWgKLME-2-rgb_front', 'bpOKSl0oIIw-8-rgb_front', '_7uBzSGPQis-3-rgb_front', 'eLv9Uhs89IQ-8-rgb_front', 'a4Nxq0QV_WA-5-rgb_front', 'EjzQn4ReeeI-5-rgb_front', 'BZqXT5UYUD8-5-rgb_front', 'fyI1Ev5m1w4-8-rgb_front', 'DKJtnSrjIro-5-rgb_front', 'D

In [ ]:
import pandas as pd

df_how2sign_test = pd.read_csv(
    "./data/how2sign_test.csv",
    sep="\t",
    dtype=str,        # optional; avoids type inference surprises
    keep_default_na=False  # optional
)


print(df_how2sign_test.columns)

Index(['VIDEO_ID', 'VIDEO_NAME', 'SENTENCE_ID', 'SENTENCE_NAME', 'START',
       'END', 'SENTENCE'],
      dtype='object')


In [3]:
df_how2sign_test.head()

,VIDEO_ID,VIDEO_NAME,SENTENCE_ID,SENTENCE_NAME,START,END,SENTENCE
0,-fZc293MpJk,-fZc293MpJk-1-rgb_front,-fZc293MpJk_0,-fZc293MpJk_0-1-rgb_front,3.33,3.68,Hi!
1,-fZc293MpJk,-fZc293MpJk-1-rgb_front,-fZc293MpJk_2,-fZc293MpJk_2-1-rgb_front,8.59,16.84,The aileron is the control surface in the wing...
2,-fZc293MpJk,-fZc293MpJk-1-rgb_front,-fZc293MpJk_3,-fZc293MpJk_3-1-rgb_front,16.84,24.81,"By moving the stick, you cause pressure to inc..."
3,-fZc293MpJk,-fZc293MpJk-1-rgb_front,-fZc293MpJk_4,-fZc293MpJk_4-1-rgb_front,27.89,35.86,The elevator is the part that moves with the s...
4,-fZc293MpJk,-fZc293MpJk-1-rgb_front,-fZc293MpJk_5,-fZc293MpJk_5-1-rgb_front,36.3,42.18,"Therefore, it's either going uphill, downhill,..."


In [5]:
# Read first 5 VIDEO_NAME values from CSV
first_5_video_names = df_how2sign_test['VIDEO_NAME'].head(5).tolist()
print("First 5 VIDEO_NAMEs:", first_5_video_names)

First 5 VIDEO_NAMEs: ['-fZc293MpJk-1-rgb_front', '-fZc293MpJk-1-rgb_front', '-fZc293MpJk-1-rgb_front', '-fZc293MpJk-1-rgb_front', '-fZc293MpJk-1-rgb_front']
